# **Live video and features**


<div style="color:#777777;margin-top: -15px;">
<b>Author</b>: Norman Juchler |
<b>Course</b>: ADLS ISP |
<b>Version</b>: v1.3 <br><br>
<!-- 23.04.2025, v1.2: Refactored text -->
<!-- 20.04.2026, v1.3: More refactoring -->
</div>

In this notebook, we take a closer look at feature detection in images — the task of identifying distinctive and informative structures such as corners, edges, or blobs. These features play a key role in applications like image matching, motion tracking, and object recognition.

We focus on corner detection and edge detection, two fundamental approaches to extracting such features. To make the tutorial a bit more fun and  interactive, we apply these techniques in real time using a live video stream from your webcam.

## **Exercises**
* [Exercise 1: Streaming a live video with OpenCV](#exercise1)     
* [Exercise 2: Canny edge detector](#exercise2)  
* [Exercise 3: Corner and blob features](#exercise3)  
* [Exercise 4: Edge detection](#exercise4)  
* [Exercise 5: Border handling](#exercise5)


---

## **Preparations**

The usual preparations... The package `isp` provides some helper functions to easily render images in this Jupyter notebook.

To access certain advanced features, we need to make sure that the opencv-contrib-python package is installed.

In [ ]:
import sys
import cv2 as cv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Enable vectorized output (for nicer plots)
%config InlineBackend.figure_formats = ["svg"]

# Inline backend configuration
%matplotlib inline

# Functionality related to this course
sys.path.append("..")
import isp

# Jupyter / IPython configuration:
# Automatically reload modules when modified, if the extension is available
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

-----

<a id='exercise1'></a>

## **Exercise 1: Streaming a live video with OpenCV**

Displaying a live video stream from your webcam is straightforward using OpenCV. The basic steps are:

* Create a `cv.VideoCapture()` object to access the camera.  
* Continuously read frames from the stream.  
* Display each frame using a suitable method.  

**Display the video:** Two approaches are shown below: one using `cv.imshow()` in a native window, and one for Jupyter notebooks using a *multithreaded* setup.

* **Option 1:** Display frames in a native window using `cv.imshow()`. This is the simplest approach and opens a separate window that updates with each new frame.  
* **Option 2:** Display the video directly inside a Jupyter notebook. This is more involved and typically requires a *multithreaded* setup: one thread captures frames from the webcam, while another updates the display. This approach offers more flexibility but can be sensitive to environment-specific limitations.  

**Camera selection:**  
The parameter `cam_id` specifies which camera to use. Typically, `0` refers to the built-in webcam, while `1` or higher selects external or virtual cameras.

Accessing hardware devices such as cameras may not work in all Jupyter environments, as it depends on system drivers and low-level access, which may be restricted. If you encounter issues, try running the code in a standalone Python environment instead.

### **Instructions:**

Review the following two examples and test whether your webcam can be used for real-time image processing.

<a id='exercise1.1'></a>

### **Example 1: Display the video in a separate window**

We define a function `run_camera_cv()` that captures a live video stream from a webcam using OpenCV, optionally applies a user-defined processing function (which we will later use to apply feature detection) to each frame, and displays the result in a window. The function runs continuously until the user presses q, ensuring proper cleanup of the camera and display resources.

In [ ]:
def run_camera_cv(cam_id, 
                  window_name="Webcam", 
                  show_ontop=True,
                  width=640,
                  height=480,
                  flip=False, 
                  func=None, 
                  **kwargs):
    """Run a camera feed and display it using OpenCV.

    Args:
        cam_id (int): Camera ID (usually 0).
        window_name (str): Name of the window.
        width (int): Width of the window.
        height (int): Height of the window.
        flip (bool): Flip the image horizontally.
        func (function): Function to apply to the image.
        **kwargs: Keyword arguments for the function.
    """

    cap = cv.VideoCapture(cam_id)

    # Adjust the camera settings (may work, or not)
    cap.set(cv.CAP_PROP_FRAME_WIDTH, width)  # adjust width
    cap.set(cv.CAP_PROP_FRAME_HEIGHT, height)  # adjust height

    # Create named window
    cv.namedWindow(window_name, cv.WINDOW_AUTOSIZE)
    # Make the window always on top
    if show_ontop:
        cv.setWindowProperty(window_name, cv.WND_PROP_TOPMOST, 1)

    try:
        while True:
            # Read image from capturing device
            success, img = cap.read()
            if not success:
                break
            if flip:
                img = cv.flip(img, 1)
            if func is not None:
                # Modify image using the provided function
                img = func(img, **kwargs)
            # Display the image in the named window
            cv.imshow(window_name, img)
            # Wait and fetch for key input (the above window should be selected)
            key = cv.waitKey(1) & 0xFF
            # Quit if "q" or "Q" is pressed.
            if key in (ord("q"), ord("Q")): 
                cap.release()
                break

    except KeyboardInterrupt:
        pass
    finally:
        # We should always release the camera
        cap.release()
        # Comment out the following lines if you want to keep the window open
        cv.destroyAllWindows() 
        cv.waitKey(1)

Next, let's demonstrate how to use the above function.

In [ ]:
####################
# Run demo
####################

# Choose the camera
cam_id = 0

# Window name
window_name = "Live Video"

# Flip the image
flip = True

# Run the camera!
run_camera_cv(cam_id, 
              window_name=window_name, 
              flip=flip)

<a id='exercise1.2'></a>

### **Example 2: Display the video directly within the Jupyter notebook**

This function also captures a live webcam stream, but displays it directly inside a Jupyter notebook using a multithreaded setup and interactive widgets. As before, it can optionally apply a user-defined processing function to each frame and allows the user to stop the stream via a button.

Note the special `**kwargs` argument: it allows you to pass additional parameters to the processing function. Any extra keyword arguments are collected in a dictionary and forwarded to the function call as `func(frame, **kwargs)`, enabling flexible customization without modifying the function signature `run_camera_jupyter()` itself.

In [ ]:
def run_camera_jupyter(cam_id=0, 
                       width=640,
                       height=480, 
                       frame_rate=0.1,
                       keep_last_frame=False,
                       func=None, 
                       **kwargs):
    """Run a camera feed and display it within Jupyter.
    
    It's nice to use this function with a Jupyter notebook, but it may be slow.

    Args: 
        cam_id: Camera ID (usually 0)
        width: Width of the image
        height: Height of the image
        frame_rate: Frame rate in frames per second
        keep_last_frame: Keep the last frame when stopping the camera
        func: Function to apply to the frame
        kwargs: Keyword arguments to pass to the function
    """

    from IPython.display import display, Image
    import ipywidgets as widgets
    import threading

    # Set up the stop button.
    stopButton = widgets.ToggleButton(
        value=False,
        description="Stop camera",
        disabled=False,
        button_style="info",  # 'success', 'info', 'warning', 'danger' or ''
        tooltip="Stop camera",
        icon="camera-retro",  # (FontAwesome names without the `fa-` prefix)
        #style=dict(font_weight="bold",)
    )

    # Display function
    import time
    def view(button):
        cap = cv.VideoCapture(cam_id)
        cap.set(cv.CAP_PROP_FRAME_WIDTH, width) # adjust width
        cap.set(cv.CAP_PROP_FRAME_HEIGHT, height) # adjust height
        
        display_handle=display(None, display_id=True)
        while True:
            _, frame = cap.read()
            frame = cv.flip(frame, 1) # if your camera reverses your image
            time.sleep(1/frame_rate)
            if frame is not None and frame.size != 0:
                if func is not None:
                    frame = func(frame, **kwargs)
                _, frame = cv.imencode(".jpeg", frame)
                image = Image(data=frame.tobytes(),
                              width=width, height=height)
                display_handle.update(image)
            if stopButton.value==True:
                print("Stopping video stream...")
                cap.release()
                if not keep_last_frame:
                    # Erase last frame
                    display_handle.update(None)
                    # Hide button
                    button.layout.visibility = "hidden"
                break
                
    # Run
    display(stopButton);
    thread = threading.Thread(target=view, args=(stopButton,));
    thread.start();


In [ ]:
####################
# Settings
####################

# Choose the camera
cam_id = 0

# Jupyter cannot process too many frames per second...
frame_rate = 10

# Adjust width and height. Use None for screen-width
width = 800
#height = None

# Keep last frame alive
keep_last_frame = False

run_camera_jupyter(cam_id, 
                   width=width, 
                   frame_rate=frame_rate, 
                   keep_last_frame=keep_last_frame);

----

Using these functions, we can stream image frames from the camera and process them in real time.

To manipulate the video stream, you can pass a custom function via the `func` argument. This function should take a single frame as input and return the processed frame. Additional parameters can be supplied via the `kwargs` dictionary. The example below demonstrates a simple function that applies a blur filter to each frame.


In [ ]:
def blur(img, kernel_size=5):
    """Blur an image using a Gaussian filter.
    """
    return cv.GaussianBlur(img, (kernel_size, kernel_size), 0) 

cam_id = 0
run_camera_cv(cam_id, 
              window_name="Demo: Blur",
              func=blur, kernel_size=105)

Alternatively, we can compute the amplitude spectrum of the image...

In [ ]:
def fft(img):
    """Compute the FFT of an image.
    """
    # Convert to grayscale
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    # Compute the FFT
    ft = scipy.fft.fft2(gray)
    ft = np.fft.fftshift(ft)
    ft_mag = np.log(np.abs(ft)+1)
    # ft_mag is float64. Normalize the image to [0,1]
    if ft_mag.max() > 0:
        ft_mag = ft_mag/ft_mag.max()
    #ft_mag = cv.normalize(ft_mag, None, 0, 1, cv.NORM_MINMAX)
    return ft_mag


run_camera_cv(cam_id, 
              window_name="Demo: Amplitude spectrum",
              func=fft)

---

<a id='exercise2'></a>

## **☆ Exercise 2: Canny edge detector**

In the previous notebook, you explored how to detect edges using the Canny edge detector. In this exercise, you will apply the same technique to a live video stream from the camera.

### **Instructions:**

Adapt the demo code above to apply Canny edge detection to each frame of the webcam stream.

In [ ]:
######################
###    EXERCISE    ###
######################
...


In [ ]:
######################
###    SOLUTION    ###
######################
cam_id = 0

# Alternative 1: Directly pass `cv.Canny` as the processing function
run_camera_cv(cam_id,
              window_name="cv.Canny()",
              func=cv.Canny,
              threshold1=50, 
              threshold2=100)

# Alternative 2: Define a wrapper function that allows us to
# apply preprocessing steps (e.g., Gaussian blur) before further processing.
def canny(img, threshold1=50, threshold2=100):
    """Apply the Canny edge detection algorithm to an image.
    """
    img = cv.GaussianBlur(img, (5, 5), 0)
    return cv.Canny(img, threshold1, threshold2)

run_camera_cv(cam_id=0, 
              window_name="Custom Canny()",
              func=canny, 
              threshold1=25, 
              threshold2=75)

---

<a id='exercise3'></a>

## **☆ Exercise 3: Corner and blob features**

Corner and blob features are distinctive points or regions in an image — for example, corners, junctions, or round blob-like structures. These features are commonly used as keypoints for tracking, matching, and object recognition.

OpenCV provides a wide range of feature detectors, including both classical and more advanced algorithms. In the lecture, you encountered (or will encounter) the Scale-Invariant Feature Transform (SIFT), but other popular methods include the Harris corner detector, the Shi–Tomasi detector, and the FAST detector. The good news is that OpenCV provides implementations for all of these, and their usage is largely consistent.


### **Instructions:**

- Skim through the following overview of feature detection algorithms available in OpenCV: [Feature detection and description](https://docs.opencv.org/4.x/db/d27/tutorial_py_table_of_contents_feature2d.html)  
- Try at least two different feature detectors and visualize their output using your webcam. If a webcam is not available, use a sample image from `../data/images`.  

In [ ]:
######################
###    EXERCISE    ###
######################

def harris_corner(img, block_size=5, ksize=3, k=0.14):
    ...
    img = ...
    return img

run_camera_cv(cam_id=0,
              window_name="Harris corner detection",
              func=harris_corner)


...
...
...

In [ ]:
######################
###    SOLUTION    ###
######################

def harris_corner(img, block_size=4, ksize=3, k=0.22):
    """Apply the Harris corner detection algorithm to an image.
    
    Parameters:
        img: Input image
        block_size: Neighborhood size
        ksize: Kernel size for the Sobel operator
        k: Harris detector free parameter (see documentation for details)
    """
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    # Detect corners
    mask = cv.cornerHarris(gray, block_size, ksize, k)
    # Visualize the corners:
    #   Step 1: Increase the size of the corner points
    #   Step 2: Apply the mask to the image
    mask = cv.dilate(mask, None)
    img[mask > 0.01*mask.max()] = [0, 0, 255]
    return img

run_camera_cv(cam_id=0,
              window_name="Harris corner detection",
              func=harris_corner)

In [ ]:
def shi_tomasi(img, max_corners=100, quality_level=0.01, min_distance=10):
    """Apply the Shi-Tomasi corner detection algorithm to an image.
    """
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    # Detect corners (as list of points)
    corners = cv.goodFeaturesToTrack(gray, max_corners, quality_level, min_distance)
    if corners is None:
        return img
    
    corners = corners.astype(int)

    # Marker color
    color = [255, 0, 255]
    
    # Draw corners on the image   
    corners = corners.astype(int)
    for i in corners:
        x, y = i.ravel()
        cv.circle(img, (x, y), 3, color, -1)
    return img

run_camera_cv(cam_id=0,
              window_name="Shi-Tomasi corner detection",
              func=shi_tomasi)

In [ ]:
def sift_features(img):
    """Apply the SIFT feature detection algorithm to an image.
    """
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    sift = cv.SIFT_create(nfeatures=50)
    #keypoints, descriptors = sift.detectAndCompute(gray, None)
    keypoints = sift.detect(gray, None)
    img = cv.drawKeypoints(gray, keypoints, img,
                           flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    return img

run_camera_cv(cam_id=0,
              window_name="SIFT feature detection",
              func=sift_features)

In [ ]:
# Not available...

# def surf_features(img, threshold=4000):
#     """Apply the SURF feature detection algorithm to an image.
#     """
#     gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
#     surf = cv.xfeatures2d.SURF_create()
#     surf.setHessianThreshold(threshold)
#     #keypoints, descriptors = surf.detectAndCompute(gray, None)
#     keypoints = surf.detect(gray, None)
#     img = cv.drawKeypoints(gray, keypoints, img,
#                            flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
#     return img

# run_camera_cv(cam_id=0,
#               window_name="SIFT feature detection",
#               func=surf_features)

In [ ]:
def fast_features(img, 
                  threshold=15, 
                  nonmax_suppression=True, 
                  mode=cv.FAST_FEATURE_DETECTOR_TYPE_9_16):
    """Apply the FAST feature detection algorithm to an image.

    Args:
        img: Input image.
        threshold: Threshold for the FAST algorithm.
        nonmaxSuppression: Apply non-maximum suppression.
        mode: Mode of the FAST algorithm, one of the following:
            - cv.FAST_FEATURE_DETECTOR_TYPE_5_8
            - cv.FAST_FEATURE_DETECTOR_TYPE_7_12
            - cv.FAST_FEATURE_DETECTOR_TYPE_9_16
    """
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    fast = cv.FastFeatureDetector_create()
    fast.setThreshold(threshold)
    fast.setNonmaxSuppression(nonmax_suppression)
    fast.setType(mode)
    keypoints = fast.detect(gray, None)
    img = cv.drawKeypoints(img, keypoints, img,
                           flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    return img

run_camera_cv(cam_id=0,
              window_name="FAST feature detection",
              func=fast_features)

In [ ]:
def brief_features(img):
    """Apply the BRIEF feature detection algorithm to an image.
    """
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    star = cv.xfeatures2d.StarDetector_create()
    brief = cv.xfeatures2d.BriefDescriptorExtractor_create()
    keypoints = star.detect(gray, None)
    keypoints, descriptors = brief.compute(gray, keypoints)
    img = cv.drawKeypoints(img, keypoints, img,
                           flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    return img

run_camera_cv(cam_id=0,
              window_name="BRIEF feature detection",
              func=brief_features)

In [ ]:
def hough_lines(img, 
                threshold=100, 
                min_line_length=50, 
                max_line_gap=10):
    """Apply the Hough line detection algorithm to an image.
    """
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    edges = cv.Canny(gray, 50, 150, apertureSize=3)
    lines = cv.HoughLinesP(edges, 1, np.pi/180, threshold, min_line_length, max_line_gap)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv.line(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    return img

run_camera_cv(cam_id=0,
              window_name="Hough line detection",
              func=hough_lines) 